In [14]:
import warnings
from urllib3.exceptions import InsecureRequestWarning
from elasticsearch import Elasticsearch
from faker import Faker


In [15]:
fake = Faker()

In [16]:
warnings.simplefilter('ignore', InsecureRequestWarning)

es = Elasticsearch(
    "https://localhost:9200",
    basic_auth=("elastic", 'Elastic_@8899'),
    verify_certs=False
    )

# es.info()

/home/el7m7/anaconda3/envs/DataEngineering/lib/python3.10/site-packages/elasticsearch/_sync/client/__init__.py:329: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [17]:
doc = {'name': fake.name(), 'street': fake.street_address(), 'city': fake.city(), 'zip': fake.zipcode()}
res = es.index(index='users', document=doc)

res['result']

'created'

## **Inserting data using helpers**

In [18]:
from elasticsearch import helpers

In [23]:
actions = [
    {
        '_index': 'users',
        '_source': {
            'name': fake.name(),
            'street': fake.street_address(),
            'city': fake.city(),
            'zip': fake.zipcode()
        }
    }
    for _ in range(999)
]

In [25]:
success_count, errors = helpers.bulk(es, actions)

print(f"Successfully indexed: {success_count} documents")
if errors:
    print(f"Errors encountered: {errors}")

Successfully indexed: 999 documents


## **Querying Elasticsearch**

In [30]:
doc = {'query': {'match_all': {}}}

In [31]:
res = es.search(index='users', body=doc, size=5)

res['hits']['hits']

/tmp/ipykernel_14858/1035700014.py:1: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  res = es.search(index='users', body=doc, size=5)


[{'_index': 'users',
  '_id': 'xB6hf6EBuQqUJeEpcU5v',
  '_score': 1.0,
  '_source': {'name': 'Jesse Oneill',
   'street': '62449 Johns Hollow Apt. 327',
   'city': 'Lake Jodi',
   'zip': '29720'}},
 {'_index': 'users',
  '_id': 'xR6tf6EBuQqUJeEpaU7V',
  '_score': 1.0,
  '_source': {'name': 'Yvonne Duran',
   'street': '967 Brandon Drive Apt. 220',
   'city': 'Mannport',
   'zip': '97863'}},
 {'_index': 'users',
  '_id': 'xh6tf6EBuQqUJeEpaU7V',
  '_score': 1.0,
  '_source': {'name': 'Lacey Martinez',
   'street': '325 Gloria Forges',
   'city': 'Anthonyville',
   'zip': '76128'}},
 {'_index': 'users',
  '_id': 'xx6tf6EBuQqUJeEpaU7V',
  '_score': 1.0,
  '_source': {'name': 'Elizabeth Gonzales',
   'street': '58030 Ryan Greens',
   'city': 'North Melaniemouth',
   'zip': '66792'}},
 {'_index': 'users',
  '_id': 'yB6tf6EBuQqUJeEpaU7V',
  '_score': 1.0,
  '_source': {'name': 'Nicholas Ford',
   'street': '39217 Ariana Forges Suite 875',
   'city': 'South Bethville',
   'zip': '49863'}}]

In [32]:
for doc in res['hits']['hits']:
    print(doc['_source'])

{'name': 'Jesse Oneill', 'street': '62449 Johns Hollow Apt. 327', 'city': 'Lake Jodi', 'zip': '29720'}
{'name': 'Yvonne Duran', 'street': '967 Brandon Drive Apt. 220', 'city': 'Mannport', 'zip': '97863'}
{'name': 'Lacey Martinez', 'street': '325 Gloria Forges', 'city': 'Anthonyville', 'zip': '76128'}
{'name': 'Elizabeth Gonzales', 'street': '58030 Ryan Greens', 'city': 'North Melaniemouth', 'zip': '66792'}
{'name': 'Nicholas Ford', 'street': '39217 Ariana Forges Suite 875', 'city': 'South Bethville', 'zip': '49863'}


In [34]:
import pandas as pd

In [46]:
df = pd.json_normalize(res['hits']['hits'])

df

,_index,_id,_score,_source.name,_source.street,_source.city,_source.zip
0,users,xB6hf6EBuQqUJeEpcU5v,1.0,Jesse Oneill,62449 Johns Hollow Apt. 327,Lake Jodi,29720
1,users,xR6tf6EBuQqUJeEpaU7V,1.0,Yvonne Duran,967 Brandon Drive Apt. 220,Mannport,97863
2,users,xh6tf6EBuQqUJeEpaU7V,1.0,Lacey Martinez,325 Gloria Forges,Anthonyville,76128
3,users,xx6tf6EBuQqUJeEpaU7V,1.0,Elizabeth Gonzales,58030 Ryan Greens,North Melaniemouth,66792
4,users,yB6tf6EBuQqUJeEpaU7V,1.0,Nicholas Ford,39217 Ariana Forges Suite 875,South Bethville,49863


In [49]:
doc = {'query': {'match': {'name': 'Veronica Wright'}}}
res = es.search(index='users', body=doc)

res['hits']['hits'][0]['_source']

{'name': 'Veronica Wright',
 'street': '427 Murphy Extensions',
 'city': 'West Charlesview',
 'zip': '50933'}

In [56]:
res= es.search(index='users', q='name:Veronica Wright', size=10)
res['hits']['hits'][0]['_source']

{'name': 'Veronica Wright',
 'street': '427 Murphy Extensions',
 'city': 'West Charlesview',
 'zip': '50933'}

In [58]:
doc={"query":{"match":{"city":"West"}}}
res=es.search(index="users", body=doc)

res['hits']['hits']

[{'_index': 'users',
  '_id': 'yh6tf6EBuQqUJeEpaU7V',
  '_score': 2.4992435,
  '_source': {'name': 'Jose Garcia',
   'street': '1668 Barbara Heights',
   'city': 'West Elizabethport',
   'zip': '18110'}},
 {'_index': 'users',
  '_id': 'zx6tf6EBuQqUJeEpaU7W',
  '_score': 2.4992435,
  '_source': {'name': 'Robert Thomas',
   'street': '3036 Michael Summit Suite 129',
   'city': 'West Melissabury',
   'zip': '80559'}},
 {'_index': 'users',
  '_id': '0h6tf6EBuQqUJeEpaU7W',
  '_score': 2.4992435,
  '_source': {'name': 'Christopher Ballard',
   'street': '0865 Vasquez Lodge Apt. 799',
   'city': 'West Ryanmouth',
   'zip': '07190'}},
 {'_index': 'users',
  '_id': '1R6tf6EBuQqUJeEpaU7W',
  '_score': 2.4992435,
  '_source': {'name': 'Leah Russo',
   'street': '0593 Harris Burg Apt. 724',
   'city': 'West Nancy',
   'zip': '17548'}},
 {'_index': 'users',
  '_id': '1h6tf6EBuQqUJeEpaU7W',
  '_score': 2.4992435,
  '_source': {'name': 'Veronica Wright',
   'street': '427 Murphy Extensions',
   'city

In [61]:
doc={"query":{"bool":{"must":{"match":{"city":"West Rhonda"}},
"filter":{"term":{"zip":"31263"}}}}}
res=es.search(index="users", body=doc)

res['hits']['hits']

[{'_index': 'users',
  '_id': '-R6tf6EBuQqUJeEpaU7X',
  '_score': 8.375574,
  '_source': {'name': 'Justin Smith',
   'street': '26609 Robinson Hill',
   'city': 'West Rhonda',
   'zip': '31263'}},
 {'_index': 'users',
  '_id': '4B6uf6EBuQqUJeEpV1Jo',
  '_score': 8.375574,
  '_source': {'name': 'Justin Smith',
   'street': '26609 Robinson Hill',
   'city': 'West Rhonda',
   'zip': '31263'}}]

## **Using scroll to handle larger results**

In [64]:
res = es.search(
    index='users',
    scroll='20m',
    size=500,
    body={
        "query": {
            "match_all": {}
        }
    }
)

/tmp/ipykernel_14858/2631944063.py:1: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  res = es.search(


In [65]:
sid = res['_scroll_id']
size = res['hits']['total']['value']

In [68]:
while (size > 0):
    res = es.scroll(scroll_id = sid, scroll = '20m')
    sid = res['_scroll_id']
    size = len(res['hits']['hits'])

    for doc in res['hits']['hits']:
        print(doc['_source'])

{'name': 'Tami Fernandez', 'street': '94548 Jennifer Cove Suite 242', 'city': 'Jamesfort', 'zip': '75564'}
{'name': 'Timothy Hamilton', 'street': '32239 Julie Extension Suite 935', 'city': 'Pruittburgh', 'zip': '77180'}
{'name': 'Blake Bell', 'street': '32832 Paige Station Apt. 191', 'city': 'North Phillip', 'zip': '86818'}
{'name': 'Susan Ortiz', 'street': '546 Eric Burgs', 'city': 'East Jessica', 'zip': '52947'}
{'name': 'Michelle Schroeder', 'street': '96226 Hernandez Mill', 'city': 'Nicholasbury', 'zip': '10558'}
{'name': 'Jennifer Good', 'street': '2660 Carlson Manor', 'city': 'West Kaylabury', 'zip': '09504'}
{'name': 'Carlos Garcia', 'street': '32667 Ruiz Ford Suite 332', 'city': 'North James', 'zip': '14680'}
{'name': 'Mark Mahoney', 'street': '309 Claudia Camp Suite 995', 'city': 'Grayview', 'zip': '90913'}
{'name': 'Micheal Richardson', 'street': '145 Phyllis Place', 'city': 'East Cynthia', 'zip': '52763'}
{'name': 'Kimberly Morales', 'street': '8797 Dorsey Well', 'city': 'No